# SearchLibrium Examples

This notebook demonstrates JAX-compatible models in SearchLibrium, including Nested Logit, Ordered Logit, Mixed models with random parameters, and Latent Class models.

In [ ]:
import numpy as np
import pandas as pd
from SearchLibrium import Parameters, call_siman
from SearchLibrium.multinomial_nested import NestedLogit
from SearchLibrium.ordered_logit import OrderedLogit
from SearchLibrium.latent_class import LatentClassMixedLogit
from SearchLibrium.RandomP import RandomParameters

## 1. Nested Logit Model (JAX-compatible)

Nested Logit groups alternatives into nests with different substitution patterns.

In [ ]:
# Load sample data
df = pd.read_csv('https://raw.githubusercontent.com/zahern/HypothesisX/main/data/Swissmetro_final.csv')
varnames = ['TIME', 'COST', 'HEADWAY']
choice_set = np.unique(df['alt']).tolist()

# Define nests
nests = {'PublicTransport': [0, 1], 'Private': [2, 3]}
lambdas = {'PublicTransport': 0.8, 'Private': 1.0}

params = Parameters(
    criterions=[('bic', -1)],
    df=df,
    varnames=varnames,
    asvarnames=varnames,
    choice_set=choice_set,
    choices=df['CHOICE'].values,
    alt_var=df['alt'].values,
    choice_id=df['custom_id'].values,
    base_alt='SM',
    models=['nested_logit'],
    nests=nests,
    lambdas=lambdas,
    p_val=0.05
)

best = call_siman(params, init_sol=None, id_num=1)
print('Nested Logit results:', best)

## 2. Ordered Logit Model (JAX-compatible)

Ordered Logit is used for ordinal dependent variables.

In [ ]:
# Create sample ordinal data
np.random.seed(42)
n = 1000
X = np.random.randn(n, 2)
beta = np.array([1.0, -0.5])
thresholds = np.array([0.0, 1.0, 2.0])  # For 4 categories
y_star = X @ beta + np.random.logistic(0, 1, n)
y = np.digitize(y_star, thresholds)

# Fit Ordered Logit
model = OrderedLogit(_jax=True, X=X, y=y, varnames=['var1', 'var2'], J=4, distr='logit')
model.fit()
model.report()

## 3. Mixed Logit with Random Parameters

All models can now include random parameters for heterogeneity.

In [ ]:
# Mixed Logit with random parameters
params = Parameters(
    criterions=[('bic', -1)],
    df=df,
    varnames=varnames,
    asvarnames=varnames,
    choice_set=choice_set,
    choices=df['CHOICE'].values,
    alt_var=df['alt'].values,
    choice_id=df['custom_id'].values,
    ind_id=df['ID'].values,
    base_alt='SM',
    models=['mixed_logit'],
    allow_random=True,
    randvars={'TIME': 'n', 'COST': 'ln'},  # Normal and log-normal distributions
    n_draws=500,
    p_val=0.05
)

best = call_siman(params, init_sol=None, id_num=1)
print('Mixed Logit with random parameters results:', best)

## 4. Latent Class Model (JAX-compatible)

Latent Class models identify unobserved heterogeneity by segmenting the population.

In [ ]:
# Prepare data for latent class
X = df[['TIME', 'COST', 'HEADWAY']].values
y = df['CHOICE'].values
alts = df['alt'].values
ids = df['ID'].values

# Fit Latent Class Mixed Logit
lc_model = LatentClassMixedLogit(
    n_classes=2,
    _jax=True,
    random_state=42
)

# Note: Latent class requires specific setup, this is a simplified example
print('Latent Class model initialized with JAX support')

## 5. Using RandomParameters Class

The RandomParameters class handles various distributions for random coefficients.

In [ ]:
# Example of using RandomParameters
distributions = ['n', 'ln', 't']  # Normal, log-normal, triangular
rand_params = RandomParameters(distributions, backend='jax')

# Generate random draws
betas_random = np.random.randn(100, 3, 500)  # 100 obs, 3 params, 500 draws
transformed = rand_params.apply_distribution(betas_random)
print('Random parameters transformed with distributions:', distributions)